# How WALRUS samples training windows, and how often they repeat

Companion to `mydocs/walrus_sampling_and_repetition.md`. Every claim tagged **[verified]** there
is checked by an `assert` here, and the section numbers (E1–E8) match the doc.

Most experiments use the **real** `BatchedMultisetSampler` and `DataLoader`, and a stand-in
dataset that only returns the indices it was asked for. That is enough, because everything that
decides *which* window is drawn happens in the sampler and the loader, not in the HDF5 reading.
Section E7 then checks the index → (file, trajectory, start frame, `dt`) mapping on a real
dataset, and E8 turns all of it into repetition estimates for the PT9 runs.

Runs on the login node on CPU. Only E7 reads data.

**Sampling scheme (branch `fulldet`).**
- Every epoch re-seeds all RNGs from `(seed, rank, epoch)` (`Trainer.train`).
- One **epoch plan**, computed identically on every rank (`epoch_plan`, `mixed_dset_sampler.py`),
  fixes which dataset every rank trains on at every step, and which part of that dataset's
  window stream it takes.
- Each dataset's window stream in an epoch is keyed on `(seed, epoch, dataset)`, with **no rank**
  in the key, and ranks take disjoint positions in it. Within an epoch no two GPUs or nodes get the
  same window until every window of the dataset has been used.

How this differs from branch `flopcount` (independent per-rank permutations) and from commit
`656c00c` (seeding from the global RNG) is set out in the history notes.

In [1]:
import itertools
import math
import sys
from collections import Counter, deque
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset, DistributedSampler, RandomSampler

REPO = Path.cwd().parent if Path.cwd().name == "demo_notebooks" else Path.cwd()
sys.path[:0] = [str(REPO / "scripts"), str(REPO / "LeonardoRunScripts" / "PT9_2d")]

from walrus.data.mixed_dset_sampler import BatchedMultisetSampler
from walrus.utils.experiment_utils import seed_all

print("torch", torch.__version__)  # the RNG details below are specific to this version's sampler/dataloader code

/scratch_local/ipykernel_995212/3044105240.py:8: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


torch 2.5.1+cu121


## E0. A stand-in for `MixedWellDataset`

`BatchedMultisetSampler` only reads three things from the dataset: `sub_dsets` (only their
`len`, the window counts), `offsets` (to turn a local index into a global one) and
`effective_batch_sizes`. `MixedWellDataset.__getitem__` turns the global indices back into
`(dataset, local indices)` with `np.searchsorted(offsets, indices[0])`
(`walrus/data/multidataset.py:374-385`). The stand-in does exactly that and returns the result
instead of loading data.

`make_loader` builds the loader exactly as `MixedWellDataModule.train_dataloader` does
(`walrus/data/multidatamodule.py:414-436`): `batch_size=None`, `collate_fn=None`, `recycle=True`.
`layout` is the trainer's `sampling_ranks` (`sampling_ranks_for`, `training.py:126`). It gives
the sampling rank of every global rank: `[0, 0, 0, 0]` for 4 GPUs under fsdp, where every GPU
trains on the same dataset at each step.

`run` mirrors the training loop: re-seed from `(seed, rank, epoch)` (as `Trainer.train` does),
`set_epoch`, then one `next()` per micro-batch. After each micro-batch it makes `jitter` calls to
`torch.randint` on the global RNG, standing in for the forward pass. The PT9 jitterer
(`FixedPatchJittererBoundaryPad`, `patch_jitterers.py:460`) makes one such call per
non-singleton axis, so 2 per forward on 2D data.

In [2]:
from walrus.utils.experiment_utils import SEED_DATASET_CHOICE, SEED_GLOBAL, SEED_WINDOWS, seed_from

FSDP4 = [0] * 4  # sampling rank of each of 4 GPUs under fsdp: one row, the same dataset on every GPU


class FakeSub:
    def __init__(self, n):
        self.n = n

    def __len__(self):
        return self.n


class FakeMixed(Dataset):
    def __init__(self, sizes):
        self.sub_dsets = [FakeSub(n) for n in sizes]
        self.offsets = [0]
        for n in sizes:
            self.offsets.append(self.offsets[-1] + n)
        self.offsets[0] = -1  # as in multidataset.py:302
        self.effective_batch_sizes = [1.0] * len(sizes)

    def __len__(self):
        return self.offsets[-1]

    def __getitem__(self, indices):
        d = int(np.searchsorted(self.offsets, indices[0], side="right") - 1)
        return {"dataset": d, "local": [i - max(self.offsets[d], 0) for i in indices]}


def make_loader(sizes, B, max_samples, seed, rank, layout=FSDP4, workers=0):
    sampler = BatchedMultisetSampler(FakeMixed(sizes), batch_size=B, seed=seed, distributed=True,
                                     max_samples=max_samples, recycle=True, drop_last=False,
                                     sampling_ranks=layout, global_rank=rank)
    return DataLoader(sampler.dataset, num_workers=workers, batch_size=None, sampler=sampler,
                      collate_fn=None)


def run(sizes, B, max_samples, epochs, seed, rank, layout=FSDP4, workers=0, jitter=2):
    # One rank of a run, epoch by epoch, as Trainer.train does it. Returns {epoch: [(dataset, [local idx])]}.
    seed_all(seed + 1_000_003 * rank)  # train.py:155 (start-up)
    loader = make_loader(sizes, B, max_samples, seed, rank, layout, workers)
    log = {}
    for e in epochs:
        seed_all(seed_from(SEED_GLOBAL, seed, rank, e))  # Trainer.train, top of each epoch
        loader.sampler.set_epoch(e)
        steps = []
        for batch in loader:
            steps.append((batch["dataset"], batch["local"]))
            for _ in range(jitter):
                torch.randint(-3, 4, ())  # forward-pass jitter draws on the global CPU RNG
        log[e] = steps
    return log

## E1. An exact model of every random draw

The whole mechanism written out as a predictor, using **no** walrus or `DataLoader` code. If it
reproduces the real loader draw for draw, each step below is exactly what happens:

1. **Dataset choice.** Each *row* has a generator seeded
   `seed_from(SEED_DATASET_CHOICE, seed, epoch, row)`, which picks uniformly over the datasets at
   every step. A row is the set of ranks that share a sampling rank, e.g. one FSDP group. Every
   rank replays every row's choices.
2. **Window stream.** Each dataset has one stream per epoch, shared by all ranks. Position `p` is
   window `perm_k[p mod W_d]` with `k = p // W_d`, where `perm_k = randperm(W_d)` seeded
   `seed_from(SEED_WINDOWS, seed, epoch, d, k)`. The rank is not in the key.
3. **Claiming.** At each step, the rows in order, and within a row its ranks in order, each take the
   next `B` positions of their dataset's stream. Two ranks never get the same position. When a
   stream runs past `W_d` it simply continues into a new permutation (`k + 1`). A micro-batch can
   straddle two permutations, and nothing is dropped.

None of this touches the global RNG. So the predictor ignores `data_workers` and the forward-pass
draws (`jitter`), and must still match for every value of them.

*History.* On branch `flopcount`, each rank had its own permutations, keyed on
`(seed, global rank, epoch, dataset)`. Ranks therefore overlapped (E5), and a recycle threw away the
partial batch and drew a new dataset. Before that (commit `656c00c`), the permutations were seeded
from the **global** CPU RNG, so the DataLoader's `_base_seed` draw, the prefetch depth and every
forward pass all shifted which windows were drawn.

In [3]:
def predict(sizes, B, max_samples, epochs, seed, rank, layout=FSDP4):
    K, rows, log = len(sizes), sorted(set(layout)), {}
    for e in epochs:
        gens = {row: torch.Generator().manual_seed(seed_from(SEED_DATASET_CHOICE, seed, e, row))
                for row in rows}                                                   # 1. dataset choice
        perms, used, steps = {}, [0] * K, []

        def window(d, p):                                                          # 2. window stream
            k, j = divmod(p, sizes[d])
            if (d, k) not in perms:
                g = torch.Generator().manual_seed(seed_from(SEED_WINDOWS, seed, e, d, k))
                perms[d, k] = torch.randperm(sizes[d], generator=g).tolist()
            return perms[d, k][j]

        for _ in range(max_samples):
            for row in rows:
                d = int(torch.randint(0, K, (1,), generator=gens[row]).item())
                for r in [r for r, rr in enumerate(layout) if rr == row]:          # 3. claim in rank order
                    batch = [window(d, p) for p in range(used[d], used[d] + B)]
                    used[d] += B
                    if r == rank:
                        steps.append((d, batch))
        log[e] = steps
    return log


# Small datasets on purpose, so streams wrap within an epoch (dataset 2 has only 25 windows).
SIZES, B, MS = [5000, 300, 25], 2, 400
LAYOUTS = {"fsdp": FSDP4, "hsdp 2x2": [0, 0, 1, 1], "ddp": [0, 1, 2, 3]}
for name, layout in LAYOUTS.items():
    for workers, jitter in [(0, 0), (2, 2), (4, 2)]:
        for rank in (0, 3):
            real = run(SIZES, B, MS, range(1, 3), 0, rank, layout, workers=workers, jitter=jitter)
            assert real == predict(SIZES, B, MS, range(1, 3), 0, rank, layout), (name, workers, jitter, rank)
print("E1: predictor == real loader, draw for draw (fsdp / hsdp / ddp layouts, 2 epochs, ranks 0 and 3, "
      "with wrapping, workers 0/2/4, jitter 0/2)")

E1: predictor == real loader, draw for draw (fsdp / hsdp / ddp layouts, 2 epochs, ranks 0 and 3, with wrapping, workers 0/2/4, jitter 0/2)


The predictor matches exactly, and it never looks at `data_workers` or the forward passes. Two
consequences:

- **A rank's windows are a pure function of `(seed, epoch, layout, rank)`.** They don't depend on
  the loader's worker count, the prefetch depth, or what the model does with the global RNG.
- **Offline replay of the windows is exact and cheap.** `epoch_plan` itself does it (that is
  what `count_WALRUS_flops.replay_epoch` calls). `dt` and the rotation are drawn in the workers
  and still depend on `data_workers`; for those, logging is still the exact route.

## E2. Which dataset: the same on every GPU of a row, and set by the seed

Under `fsdp` the sampling rank is 0 on every GPU (`sampling_ranks_for`, `training.py:126`), so all
GPUs share the dataset sequence. It is keyed on `(seed, epoch)`, so it changes with the epoch
**and with `cfg.seed`**. It matches `count_WALRUS_flops.replay_epoch`, which sums the plan over
all ranks.

*History:* before the per-epoch seeding, the generator was seeded `100·epoch + 10·0 + rank`.
The sampler's `seed` was never passed, so every seed shared one dataset order, and the linear
formula collided (seed 10, epoch 1 = seed 0, epoch 2).

In [4]:
import count_WALRUS_flops as cwf

SIZES9 = [9000] * 9
seqs = {}
for seed, rank in [(0, 0), (0, 1), (0, 3), (1, 0), (1, 2)]:
    log = run(SIZES9, 2, 500, [1, 2], seed, rank, workers=2)
    seqs[(seed, rank)] = {e: [d for d, _ in log[e]] for e in log}
assert seqs[(0, 0)] == seqs[(0, 1)] == seqs[(0, 3)]   # all GPUs, same seed
assert seqs[(1, 0)] == seqs[(1, 2)]
assert seqs[(0, 0)] != seqs[(1, 0)]                   # the seed changes the dataset order
assert seqs[(0, 0)][1] != seqs[(0, 0)][2]             # and so does the epoch
for seed in (0, 1):
    for e in (1, 2):
        counts = Counter(seqs[(seed, 0)][e])
        assert [4 * counts[d] for d in range(9)] == cwf.replay_epoch(seed, e, FSDP4, [2] * 9, SIZES9, 500, True)
print("E2: dataset sequence shared by all ranks, set by (seed, epoch), equal to replay_epoch")

E2: dataset sequence shared by all ranks, set by (seed, epoch), equal to replay_epoch


## E3. Within an epoch, across epochs, and when a dataset runs out

- **Within an epoch**, all ranks share a dataset's stream, so no window repeats on *any* rank
  until all `W_d` windows have been used. The stream then wraps into a new permutation, and every
  window is used once more before any is used a third time.
- **Each epoch** starts new permutations. They don't continue the previous epoch's, so an epoch
  that draws only part of a dataset does *not* make the next epoch draw a different part.
- **When a dataset runs out** mid-batch, nothing is dropped: the micro-batch takes the rest of its
  windows from the next permutation.

In [5]:
SIZES, B, MS, R = [5000, 300, 25], 2, 400, 4
logs = {r: run(SIZES, B, MS, [1, 2], 0, r, workers=2) for r in range(R)}


def stream(e, d):
    # Dataset d's windows in the order they were claimed: step by step, ranks 0..3 (one fsdp row).
    return [i for t in range(MS) for r in range(R) if logs[r][e][t][0] == d for i in logs[r][e][t][1]]


# Dataset 0 (5000 windows) never runs out: no window repeats within an epoch, across all 4 ranks.
big1, big2 = stream(1, 0), stream(2, 0)
assert len(big1) == len(set(big1)) and len(big2) == len(set(big2))
overlap = len(set(big1) & set(big2))
print(f"dataset 0 (5000 windows): {len(big1)} draws over 4 ranks in epoch 1, {len(big2)} in epoch 2, all distinct; "
      f"{overlap} windows in common between the epochs (independent subsets expect ≈ {len(big1) * len(big2) / 5000:.0f})")

# Dataset 2 (25 windows) wraps many times per epoch. Every block of 25 consecutive stream positions
# is a permutation of all 25 windows: nothing dropped, each pass a new order.
small = stream(1, 2)
full = [small[i:i + 25] for i in range(0, len(small) - 24, 25)]
assert all(sorted(b) == list(range(25)) for b in full)
assert len({tuple(b) for b in full}) == len(full)
print(f"dataset 2 (25 windows): {len(small)} draws over 4 ranks in epoch 1 = {len(full)} full passes "
      f"(+ {len(small) - 25 * len(full)}), each a new permutation of all 25 windows")

dataset 0 (5000 windows): 984 draws over 4 ranks in epoch 1, 1032 in epoch 2, all distinct; 202 windows in common between the epochs (independent subsets expect ≈ 203)
dataset 2 (25 windows): 1216 draws over 4 ranks in epoch 1 = 48 full passes (+ 16), each a new permutation of all 25 windows


## E4. Forward passes no longer change which windows are drawn

Same seed and same rank, with 0 vs. 2 jitter draws per micro-batch, and with `workers` of 0 or 4.
Every epoch's draws must be identical.

*History:* before the per-epoch seeding, this experiment showed the opposite. Epoch 1 was
identical only for datasets seeded within the prefetch window, and every later epoch
differed, because the forward passes consumed the global RNG that seeded the permutations.

In [6]:
SIZES3 = [5000, 4000, 3000]
logs = [run(SIZES3, 2, 300, [1, 2, 3], 0, 0, workers=w, jitter=j) for w in (0, 4) for j in (0, 2)]
assert all(lg == logs[0] for lg in logs)
print("E4: identical draws for workers 0/4 and 0/2 forward draws per step, in every epoch")

E4: identical draws for workers 0/4 and 0/2 forward draws per step, in every epoch


## E5. Across GPUs (and nodes): disjoint within an epoch

Ranks take disjoint positions of a shared stream, so within an epoch two ranks share a window only
once the epoch's total demand `N` for that dataset exceeds `W`. Even then, every window is used
`⌊N/W⌋` or `⌈N/W⌉` times. Nodes are just more ranks, so the same holds across nodes. The check
runs the fsdp, HSDP and DDP layouts. Under HSDP and DDP, rows draw different datasets at each step
and so use up a stream at different rates. Every rank replays all rows' choices (E1), so the
positions still never collide.

The last column is what branch `flopcount`'s independent per-rank permutations would give: an
expected pairwise overlap of `r²/W` (hypergeometric mean), where `r` is a rank's draws.

In [7]:
SIZES2, B, MS = [3000, 50000], 2, 2000
rows = []
for name, layout in LAYOUTS.items():
    R = len(layout)
    logs = [run(SIZES2, B, MS, [1], 0, r, layout, jitter=0)[1] for r in range(R)]
    for d, W in enumerate(SIZES2):
        uses = Counter(i for lg in logs for dd, bb in lg if dd == d for i in bb)
        counts = [uses[i] for i in range(W)]
        assert max(counts) - min(counts) <= 1, (name, d)   # every window used before any is reused
        sets = [{i for dd, bb in lg if dd == d for i in bb} for lg in logs]
        r = np.mean([sum(len(bb) for dd, bb in lg if dd == d) for lg in logs])
        pair = np.mean([len(x & y) for x, y in itertools.combinations(sets, 2)])
        if sum(counts) <= W:
            assert pair == 0
        rows.append({"layout": name, "W": W, "draws N (4 ranks)": sum(counts),
                     "uses per window": f"{min(counts)}–{max(counts)}", "pair overlap": pair,
                     "flopcount: expected r²/W": r * r / W})
pd.DataFrame(rows).round(1)

,layout,W,draws N (4 ranks),uses per window,pair overlap,flopcount: expected r²/W
0,fsdp,3000,7992,2–3,695.7,1330.7
1,fsdp,50000,8008,0–1,0.0,80.2
2,hsdp 2x2,3000,8116,2–3,708.7,1372.3
3,hsdp 2x2,50000,7884,0–1,0.0,77.7
4,ddp,3000,8036,2–3,694.2,1345.4
5,ddp,50000,7964,0–1,0.0,79.3


## E6. Why not `allow_sharding_in_train=True` (`DistributedSampler`)?

That is the built-in way to split data across ranks. On `fulldet` it is ignored, because the
epoch plan already gives disjoint windows. Branch `flopcount`'s version of this experiment showed
that it is disjoint within a pass, but:
- **Fixed slices.** Each rank owns a fixed `1/R` slice. Under HSDP or DDP, rows use up a dataset at
  different rates, so a rank that empties its slice repeats it while other slices still hold
  unused windows.
- **Recycling.** A recycle within an epoch replays the same order.
- **Padding.** `drop_last=False` pads the slices with duplicate windows.

The shared plan has none of these: E3 shows that a wrap starts a new permutation, and E5 shows the
windows stay disjoint under HSDP and DDP.

## E6b. A resumed epoch is identical to the uninterrupted one

A resumed job starts from the checkpoint's epoch. The epoch plan is keyed on `(seed, epoch)` and the
layout, and the other random streams on `(seed, rank, epoch)`. So a job that resumes at epoch `k`
draws exactly what the uninterrupted run drew at epoch `k`. The demo uses the 9 PT9 datasets at
their real window counts and `workers = 10`: job 1 runs epochs 1–5, and job 2 starts fresh (as a
resume does) at epoch 4.

*History:* checkpoints store no RNG state, and the global RNG used to be seeded only at start-up.
A resumed job therefore restarted every stream from the seed, and its first epoch partly
**replayed epoch 1**: on average 0.9 datasets per resume (at least one in 61 % of resume
epochs) redrew their epoch-1 windows, in the same order. The full trainer-level check is
`tests/test_determinism.py`, and its GPU counterpart is `LeonardoRunScripts/PT9_2d/Determinism/`.

In [8]:
import well_registry as wr
from pt9_substrate import PRETRAIN_DATASETS

reg = wr.load_registry(REPO / "demo_notebooks" / "data" / "dataset_table_2d.json")["datasets"]
W9 = [wr.n_windows(reg[n]["splits"]["train"], 6, 1, 1) for n in PRETRAIN_DATASETS]
B, MS, WORKERS = 2, 2000, 10

for rank in (0, 3):
    job1 = run(W9, B, MS, range(1, 6), 0, rank, workers=WORKERS)
    job2 = run(W9, B, MS, range(4, 6), 0, rank, workers=WORKERS)  # fresh process, resumes at epoch 4
    assert job2[4] == job1[4] and job2[5] == job1[5]
    assert job2[4] != job1[1]                                    # no replay of epoch 1
print("E6b: resumed epochs 4-5 equal the uninterrupted epochs 4-5 on ranks 0 and 3")

E6b: resumed epochs 4-5 equal the uninterrupted epochs 4-5 on ranks 0 and 3


## E7. From a global index to (file, trajectory, start frame, `dt`) on real data

`the_well/data/datasets.py:780-813`, `_load_one_sample(index)`:

```
file_idx   = searchsorted(file_index_offsets, index, side="right") - 1
local      = index - file_index_offsets[file_idx]          # offsets[0] is -1, clamped to 0
traj       = local // windows_per_trajectory[file_idx]       # = n_steps - (T_in + T_out) + 1
t0         = local %  windows_per_trajectory[file_idx]
dt         = randint(1, min(max_dt, (n_steps - t0 - 1) // (T_in + T_out - 1)) + 1)   # in the worker, np.random
```

Checked on VE, the one dataset whose files have different trajectory lengths (20 and 60 steps).
The hand mapping uses only the per-file list from the registry.

In [9]:
from walrus.data.inflated_dataset import InflatedWellDataset

ROOT = Path(wr.load_registry(REPO / "demo_notebooks" / "data" / "dataset_table_2d.json")["root"])
ve = InflatedWellDataset(path=str(ROOT / "viscoelastic_instability"), well_split_name="train",
                         n_steps_input=6, n_steps_output=1, min_dt_stride=1, max_dt_stride=5,
                         pad_cartesian_data_to_d=3)
files = reg["viscoelastic_instability"]["splits"]["train"]["files"]
assert [Path(p).name for p in ve.files_paths] == [f["name"] for f in files]


def by_hand(index):
    for f_idx, f in enumerate(files):
        w = f["n_steps"] - 7 + 1
        if index < f["n_traj"] * w:
            return f_idx, index // w, index % w
        index -= f["n_traj"] * w


assert len(ve) == sum(f["n_traj"] * (f["n_steps"] - 6) for f in files)
rng = np.random.default_rng(0)
probe = sorted(set(rng.integers(0, len(ve), 12).tolist()) | {0, len(ve) - 1})
rows = []
for i in probe:
    _, f_idx, traj, t0, dt = ve._load_one_sample(i)
    assert (f_idx, traj, t0) == by_hand(i)
    rows.append({"index": i, "file": files[f_idx]["name"], "n_steps": files[f_idx]["n_steps"],
                 "traj": traj, "t0": t0, "dt": dt})
pd.DataFrame(rows)

,index,file,n_steps,traj,t0,dt
0,0,viscoelastic_instability_AH.hdf5,20,0,0,1
1,89,viscoelastic_instability_AH.hdf5,20,6,5,1
2,222,viscoelastic_instability_AH.hdf5,20,15,12,1
3,407,viscoelastic_instability_CAR.hdf5,60,3,21,5
4,950,viscoelastic_instability_CAR.hdf5,60,13,24,4
5,1462,viscoelastic_instability_CAR.hdf5,60,22,50,1
6,1669,viscoelastic_instability_CAR.hdf5,60,26,41,3
7,2771,viscoelastic_instability_EIT.hdf5,60,14,9,2
8,3453,viscoelastic_instability_EIT.hdf5,60,26,43,1
9,3521,viscoelastic_instability_transtochaos_EIT_L.hdf5,20,0,3,2


In [10]:
from the_well.data.utils import maximum_stride_for_initial_index

# dt depends on the start frame: early windows can stretch up to dt=5, late ones are forced to dt=1.
f60 = next(i for i, f in enumerate(files) if f["n_steps"] == 60)
base_index = sum(f["n_traj"] * (f["n_steps"] - 6) for f in files[:f60])
rows = []
for t0 in (0, 20, 35, 45, 53):
    cap = min(5, maximum_stride_for_initial_index(t0, 60, 6, 1))
    dts = Counter(ve._load_one_sample(base_index + t0)[4] for _ in range(12))
    assert set(dts) <= set(range(1, cap + 1))
    rows.append({"t0": t0, "allowed dt": f"1..{cap}", "observed (12 loads)": dict(sorted(dts.items()))})
pd.DataFrame(rows)

,t0,allowed dt,observed (12 loads)
0,0,1..5,"{1: 1, 2: 3, 3: 3, 4: 4, 5: 1}"
1,20,1..5,"{2: 2, 3: 5, 4: 2, 5: 3}"
2,35,1..4,"{1: 3, 2: 4, 3: 4, 4: 1}"
3,45,1..2,"{1: 5, 2: 7}"
4,53,1..1,{1: 12}


## E8. Estimating repetition for the PT9 runs

From E1–E5: under fsdp, in epoch `e`, the `R` ranks together draw `N_{d,e} = R·B·c_{d,e}` windows
of dataset `d`, where `c_{d,e}` is its number of micro-batches per rank (the same on every rank,
E2). These are the first `N_{d,e}` positions of the epoch's shared stream:
- if `N_{d,e} ≤ W`, a uniform random subset with no repeats;
- otherwise every window is used `⌊N/W⌋` or `⌈N/W⌉` times.

Epochs are independent. Over `E` epochs, the number of times a given window is drawn is then
approximately Poisson with mean `λ = N/W`, where `N = Σ_e N_{d,e}`. That gives:

| quantity | formula |
|---|---|
| draws `N_d` | `Σ_e R·B·c_{d,e}` |
| distinct windows drawn | `W·[1 − Π_e (1 − min(N_{d,e}, W)/W)]` |
| exact-repeat draws | `N − distinct` (draws of a window already drawn before) |
| share of windows drawn ≥ 2 times | `≈ 1 − e^{−λ}(1 + λ)` |
| same for `flopcount` (independent ranks) | distinct `= W·[1 − Π_e (1 − r_{d,e}/W)^R]`, with `r = N/R` |

The gain over `flopcount` is inside each epoch. Across epochs both schemes draw independent
subsets, so with `N_{d,e} ≪ W` the two agree.

**Frame overlap.** Two different windows of the same trajectory share input frames when their
start frames are close (with `dt = 1`, `|t0 − t0'| ≤ 5`). That is not countable in closed form
once `dt` varies, so it is simulated. Draws are made in the order of each epoch's stream, each
with a `dt` drawn by the real rule, and a draw counts as "frame-overlapping" if any of its 6 input
frames was an input frame of an earlier draw.

In [11]:
CODE = {"acoustic_scattering_discontinuous": "AD", "acoustic_scattering_inclusions": "AI",
        "acoustic_scattering_maze": "AM", "euler_multi_quadrants_openBC": "EMQ-O",
        "euler_multi_quadrants_periodicBC": "EMQ-P", "gray_scott_reaction_diffusion": "GS",
        "planetswe": "PS", "rayleigh_benard": "RB", "shear_flow": "SF", "active_matter": "ACT",
        "helmholtz_staircase": "HS", "turbulent_radiative_layer_2D": "TRL2", "viscoelastic_instability": "VE"}
B, T_IN, T_OUT, MAX_DT, MS, E = 2, 6, 1, 5, 2000, 100


def window_table(name):
    # Map window index -> (trajectory id, t0, n_steps) for a whole train split.
    traj, t0, nst, base = [], [], [], 0
    for g in reg[name]["splits"]["train"]["steps"]:
        w = g["n_steps"] - (T_IN + T_OUT) + 1
        idx = np.arange(g["n_traj"] * w)
        traj.append(base + idx // w); t0.append(idx % w); nst.append(np.full(idx.size, g["n_steps"]))
        base += g["n_traj"]
    return np.concatenate(traj), np.concatenate(t0), np.concatenate(nst), base


def simulate(name, per_epoch_draws, seed=0, block=8):
    # Exact-repeat and frame-overlap fractions over a run. per_epoch_draws[e] = N_{d,e}, all ranks together.
    rng = np.random.default_rng(seed)
    traj, t0, nst, n_traj = window_table(name)
    W = traj.size
    seen_w = np.zeros(W, bool)
    seen_f = np.zeros((n_traj, int(nst.max())), bool)
    rep = ovl = total = 0
    for n_e in per_epoch_draws:
        # The epoch's shared stream: permutations of all W windows back to back, cut at n_e.
        idx = np.concatenate([rng.permutation(W) for _ in range(-(-n_e // W))])[:n_e]
        cap = np.minimum((nst[idx] - t0[idx] - 1) // (T_IN + T_OUT - 1), MAX_DT)
        dt = np.where(cap > 1, rng.integers(1, np.maximum(cap, 1) + 1), 1)
        frames = t0[idx, None] + dt[:, None] * np.arange(T_IN)
        tr = traj[idx]
        for s in range(0, n_e, block):  # small blocks so overlaps between nearby draws are counted
            sl = slice(s, s + block)
            rep += seen_w[idx[sl]].sum()
            ovl += seen_f[tr[sl, None], frames[sl]].any(axis=1).sum()
            seen_w[idx[sl]] = True
            seen_f[tr[sl, None], frames[sl]] = True
        total += n_e
    return rep / total, ovl / total, seen_w.mean(), seen_f[:, :].sum() / sum(
        g["n_traj"] * g["n_steps"] for g in reg[name]["splits"]["train"]["steps"])


def estimates(names, R, epochs=E, K=None):
    K = K or len(names)
    windows = [wr.n_windows(reg[n]["splits"]["train"], T_IN, T_OUT, 1) for n in names]
    # Micro-batches per dataset and epoch, summed over the R fsdp ranks: (E, K)
    per_epoch = np.array([cwf.replay_epoch(0, e, [0] * R, [B] * K, windows, MS, True) for e in range(1, epochs + 1)])
    rows = []
    for d, n in enumerate(names):
        W, n_e = windows[d], B * per_epoch[:, d]
        N = n_e.sum()
        lam = N / W
        distinct = W * (1 - np.prod(1 - np.minimum(n_e, W) / W))
        distinct_fc = W * (1 - np.prod((1 - np.minimum(n_e / R, W) / W) ** R))
        rep_sim, ovl_sim, cov_sim, frame_cov = simulate(n, n_e)
        rows.append({
            "dataset": CODE[n], "W": W, "draws/epoch (all ranks)": n_e.mean(), "λ = N/W": lam,
            "window cov.": distinct / W, "cov. (sim)": cov_sim,
            "exact-repeat draws": 1 - distinct / N, "repeat (sim)": rep_sim,
            "repeat (flopcount)": 1 - distinct_fc / N,
            "windows drawn ≥2×": 1 - math.exp(-lam) * (1 + lam),
            "frame-overlap draws (sim)": ovl_sim,
        })
        assert abs(rows[-1]["window cov."] - cov_sim) < 0.01 and abs(rows[-1]["exact-repeat draws"] - rep_sim) < 0.01
    return pd.DataFrame(rows).set_index("dataset")


FMT = {"W": "{:,}", "draws/epoch (all ranks)": "{:.0f}", "λ = N/W": "{:.2f}", "window cov.": "{:.1%}",
       "cov. (sim)": "{:.1%}", "exact-repeat draws": "{:.1%}", "repeat (sim)": "{:.1%}",
       "repeat (flopcount)": "{:.1%}", "windows drawn ≥2×": "{:.1%}", "frame-overlap draws (sim)": "{:.1%}"}
est_n1 = estimates(PRETRAIN_DATASETS, R=4)
est_n1.style.format(FMT).set_caption("PT9 N1: 4 GPUs, fsdp, 9 datasets, 100 epochs")

,W,draws/epoch (all ranks),λ = N/W,window cov.,cov. (sim),exact-repeat draws,repeat (sim),repeat (flopcount),windows drawn ≥2×,frame-overlap draws (sim)
dataset,,,,,,,,,,
AD,"153,600",1776,1.16,68.7%,68.9%,40.5%,40.4%,40.7%,32.2%,95.8%
AI,"268,608",1767,0.66,48.3%,48.3%,26.6%,26.6%,26.7%,14.1%,92.5%
AM,"313,600",1773,0.57,43.3%,43.3%,23.5%,23.5%,23.6%,11.1%,91.8%
EMQ-O,"380,000",1755,0.46,37.0%,37.0%,19.8%,19.9%,19.9%,7.9%,89.4%
EMQ-P,"380,000",1792,0.47,37.7%,37.7%,20.1%,20.2%,20.2%,8.2%,89.6%
GS,"955,200",1775,0.19,17.0%,17.0%,8.7%,8.7%,8.7%,1.5%,76.6%
PS,"96,192",1782,1.85,84.6%,84.7%,54.3%,54.3%,54.5%,55.3%,97.6%
RB,"271,600",1793,0.66,48.4%,48.4%,26.6%,26.7%,26.8%,14.2%,93.0%
SF,"173,824",1786,1.03,64.4%,64.5%,37.3%,37.3%,37.5%,27.4%,95.5%


In [12]:
est_n2 = estimates(PRETRAIN_DATASETS, R=8)
est_n2.style.format(FMT).set_caption("PT9 N2: 8 GPUs (2 nodes), fsdp, 9 datasets, 100 epochs")

,W,draws/epoch (all ranks),λ = N/W,window cov.,cov. (sim),exact-repeat draws,repeat (sim),repeat (flopcount),windows drawn ≥2×,frame-overlap draws (sim)
dataset,,,,,,,,,,
AD,"153,600",3552,2.31,90.4%,90.4%,60.9%,60.9%,61.0%,67.2%,97.9%
AI,"268,608",3534,1.32,73.4%,73.4%,44.2%,44.2%,44.4%,37.9%,96.3%
AM,"313,600",3546,1.13,67.9%,67.9%,39.9%,40.0%,40.1%,31.2%,95.9%
EMQ-O,"380,000",3509,0.92,60.5%,60.4%,34.5%,34.6%,34.7%,23.6%,94.7%
EMQ-P,"380,000",3585,0.94,61.2%,61.3%,35.1%,35.0%,35.2%,24.3%,94.7%
GS,"955,200",3550,0.37,31.1%,31.1%,16.3%,16.3%,16.5%,5.4%,88.1%
PS,"96,192",3564,3.71,97.7%,97.7%,73.6%,73.6%,73.7%,88.4%,98.8%
RB,"271,600",3586,1.32,73.5%,73.5%,44.3%,44.3%,44.5%,38.0%,96.5%
SF,"173,824",3573,2.06,87.5%,87.4%,57.4%,57.5%,57.6%,60.9%,97.7%


In [13]:
HELDOUT = ["active_matter", "helmholtz_staircase", "turbulent_radiative_layer_2D", "viscoelastic_instability"]
est_ft = pd.concat([estimates([n], R=4) for n in HELDOUT])
est_ft.style.format(FMT).set_caption("Held-out datasets finetuned alone: 4 GPUs, fsdp, PT9 knobs, 100 epochs")

,W,draws/epoch (all ranks),λ = N/W,window cov.,cov. (sim),exact-repeat draws,repeat (sim),repeat (flopcount),windows drawn ≥2×,frame-overlap draws (sim)
dataset,,,,,,,,,,
ACT,"13,125",16000,121.90,100.0%,100.0%,99.2%,99.2%,99.2%,100.0%,100.0%
HS,"18,304",16000,87.41,100.0%,100.0%,98.9%,98.9%,98.9%,100.0%,99.9%
TRL2,"6,840",16000,233.92,100.0%,100.0%,99.6%,99.6%,99.6%,100.0%,100.0%
VE,"5,422",16000,295.09,100.0%,100.0%,99.7%,99.7%,99.7%,100.0%,100.0%


For the single-dataset finetunes, every micro-batch comes from the one dataset, so the 4 ranks
together draw `R · B · max_samples = 16,000` windows per epoch. That is more than `W` for ACT, TRL2
and VE, so their streams wrap and every epoch uses *every* window `⌊N/W⌋` or `⌈N/W⌉` times (ACT 1–2,
TRL2 and VE 2–3). HS has 18,304 windows, so it doesn't wrap: each epoch uses 16,000 distinct
windows. `flopcount` instead drew four independent 4,000-window subsets, which left part of each
dataset unseen in the epoch while drawing other windows up to four times.

In [14]:
# A stream wraps within an epoch when R * B * (micro-batches of d per rank-epoch) > W_d.
R = 4
for n in HELDOUT:
    W = wr.n_windows(reg[n]["splits"]["train"], T_IN, T_OUT, 1)
    N = R * B * MS
    print(f"{CODE[n]:5s} W={W:6,}  draws/epoch (4 ranks)={N:,}  wraps within an epoch: {N > W}"
          f"  → uses per window per epoch: {N // W}–{-(-N // W)}")

ACT   W=13,125  draws/epoch (4 ranks)=16,000  wraps within an epoch: True  → uses per window per epoch: 1–2
HS    W=18,304  draws/epoch (4 ranks)=16,000  wraps within an epoch: False  → uses per window per epoch: 0–1
TRL2  W= 6,840  draws/epoch (4 ranks)=16,000  wraps within an epoch: True  → uses per window per epoch: 2–3
VE    W= 5,422  draws/epoch (4 ranks)=16,000  wraps within an epoch: True  → uses per window per epoch: 2–3
